# Notebook 02 — Pipeline - Limpieza y Preparación de Datos

**Proyecto:** Sistema de alerta temprana de riesgo académico estudiantil  
**Dataset:** Student Performance (student-por.csv) — UCI ML Repository  
**Objetivo:** Preparar el dataset para el entrenamiento del modelo de clasificación binaria  
**Variable objetivo:** riesgo (1 = en riesgo de reprobar, 0 = sin riesgo)

In [9]:
# Importamos las librerías que vamos a usar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import os

# Cargamos el dataset original
# Debemos recordar que el dataset usa ";" como separador, no coma ","
df = pd.read_csv("student-por.csv", sep=";")

# Confirmamos que cargó bien
print("✅ Dataset cargado correctamente")
print(f"   Filas: {df.shape[0]}")
print(f"   Columnas: {df.shape[1]}")
print(f"\nPrimeras 3 filas:")
df.head(3)

✅ Dataset cargado correctamente
   Filas: 649
   Columnas: 33

Primeras 3 filas:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12


## 1. Diagnóstico - Calidad del dataset

In [8]:
# Verificaremos la calidad del dataset
# Confirmaremos que el dataset está limpio antes de procesarlo

print("=" * 50)
print("DIAGNÓSTICO DE CALIDAD")
print("=" * 50)

print(f"\n📊 Forma del dataset: {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\n🔍 Valores nulos por columna:")
nulos = df.isnull().sum()
print(f"   Total de nulos: {nulos.sum()}")
if nulos.sum() == 0:
    print("   ✅ Sin valores nulos — no requiere imputación")

print(f"\n🔍 Filas duplicadas:")
duplicados = df.duplicated().sum()
print(f"   Total de duplicados: {duplicados}")
if duplicados == 0:
    print("   ✅ Sin duplicados — no requiere eliminación")

print(f"\n📋 Tipos de variables:")
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f"   Numéricas ({len(numericas)}): {numericas}")
print(f"   Categóricas ({len(categoricas)}): {categoricas}")

DIAGNÓSTICO DE CALIDAD

📊 Forma del dataset: 649 filas x 34 columnas

🔍 Valores nulos por columna:
   Total de nulos: 0
   ✅ Sin valores nulos — no requiere imputación

🔍 Filas duplicadas:
   Total de duplicados: 0
   ✅ Sin duplicados — no requiere eliminación

📋 Tipos de variables:
   Numéricas (17): ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3', 'riesgo']
   Categóricas (17): ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


## 2. Creación de la variable objetivo

In [4]:
# Creamos la variable objetivo "riesgo"
# Si G3 (nota final) es menor a 10 → en riesgo (1)
# Si G3 es 10 o más → sin riesgo (0)
# Esto sigue el criterio de aprobación del sistema educativo portugués

df["riesgo"] = (df["G3"] < 10).astype(int)

# Verificamos la distribución
print("=" * 50)
print("VARIABLE OBJETIVO: riesgo")
print("=" * 50)
conteo = df["riesgo"].value_counts()
porcentaje = df["riesgo"].value_counts(normalize=True) * 100

print(f"\n  Sin riesgo (0): {conteo[0]} estudiantes ({porcentaje[0]:.1f}%)")
print(f"  Con riesgo (1): {conteo[1]} estudiantes ({porcentaje[1]:.1f}%)")
print(f"\n⚠️  Desbalance de clases detectado: {porcentaje[0]:.1f}% vs {porcentaje[1]:.1f}%")
print("   → Justifica el uso de F1-score como métrica principal")

# Eliminamos G3 del dataset procesado
# (no la usamos como variable de entrada para evitar fuga de datos)
# También eliminamos G2 por alta correlación con G3 (0.92)
df_procesado = df.drop(columns=["G3", "G2"])
print(f"\n✅ Variables G3 y G2 eliminadas para evitar fuga de datos")
print(f"   Dataset procesado: {df_procesado.shape[0]} filas x {df_procesado.shape[1]} columnas")

VARIABLE OBJETIVO: riesgo

  Sin riesgo (0): 549 estudiantes (84.6%)
  Con riesgo (1): 100 estudiantes (15.4%)

⚠️  Desbalance de clases detectado: 84.6% vs 15.4%
   → Justifica el uso de F1-score como métrica principal

✅ Variables G3 y G2 eliminadas para evitar fuga de datos
   Dataset procesado: 649 filas x 32 columnas


## 3. Encoding de variables categóricas

In [5]:
# Identificamos las columnas categóricas (texto) que hay que convertir a números
categoricas = df_procesado.select_dtypes(include=['object']).columns.tolist()
print(f"Variables categóricas a codificar ({len(categoricas)}):")
for col in categoricas:
    valores = df_procesado[col].unique()
    print(f"   {col}: {valores}")

# Aplicamos Label Encoding a cada variable categórica
# LabelEncoder convierte texto a números: 'yes'→1, 'no'→0, etc.
le = LabelEncoder()
df_codificado = df_procesado.copy()  # Hacemos una copia para no modificar el original

for col in categoricas:
    df_codificado[col] = le.fit_transform(df_procesado[col])

print(f"\n✅ Encoding completado")
print(f"   Variables codificadas: {len(categoricas)}")
print(f"\nVerificación — primeras 3 filas del dataset codificado:")
df_codificado[categoricas].head(3)

Variables categóricas a codificar (17):
   school: ['GP' 'MS']
   sex: ['F' 'M']
   address: ['U' 'R']
   famsize: ['GT3' 'LE3']
   Pstatus: ['A' 'T']
   Mjob: ['at_home' 'health' 'other' 'services' 'teacher']
   Fjob: ['teacher' 'other' 'services' 'health' 'at_home']
   reason: ['course' 'other' 'home' 'reputation']
   guardian: ['mother' 'father' 'other']
   schoolsup: ['yes' 'no']
   famsup: ['no' 'yes']
   paid: ['no' 'yes']
   activities: ['no' 'yes']
   nursery: ['yes' 'no']
   higher: ['yes' 'no']
   internet: ['no' 'yes']
   romantic: ['no' 'yes']

✅ Encoding completado
   Variables codificadas: 17

Verificación — primeras 3 filas del dataset codificado:


,school,sex,address,famsize,Pstatus,Mjob,Fjob,reason,guardian,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic
0,0,0,1,0,0,0,4,0,1,1,0,0,0,1,1,0,0
1,0,0,1,0,1,0,2,0,0,0,1,0,0,0,1,1,0
2,0,0,1,1,1,0,2,2,1,1,0,0,0,1,1,1,0


## 4. Separación de variables de entrada (X) y objetivo (y)

In [6]:
# Definimos las variables de entrada (X) y la variable objetivo (y)
# X = todo lo que el modelo usará para aprender
# y = lo que el modelo debe predecir (riesgo: 0 o 1)

# Variable objetivo
y = df_codificado["riesgo"]

# Variables de entrada: todo menos la variable objetivo
# También excluimos G1 de las variables de entrada
# si queremos un modelo más "temprano" (sin notas de periodos)
# Pero como ya tenemos G1 como variable válida (nota del primer periodo),
# la incluimos — es información disponible tempranamente

X = df_codificado.drop(columns=["riesgo"])

print("=" * 50)
print("SEPARACIÓN X / y")
print("=" * 50)
print(f"\n📥 Variables de entrada (X): {X.shape[1]} columnas")
print(f"   {list(X.columns)}")
print(f"\n🎯 Variable objetivo (y): riesgo")
print(f"   Valores únicos: {y.unique()}")
print(f"   Distribución: {dict(y.value_counts())}")

SEPARACIÓN X / y

📥 Variables de entrada (X): 31 columnas
   ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1']

🎯 Variable objetivo (y): riesgo
   Valores únicos: [0 1]
   Distribución: {0: np.int64(549), 1: np.int64(100)}


## 5. División train / test

In [7]:
# Dividimos los datos en entrenamiento (80%) y prueba (20%)
# random_state=42 garantiza que siempre salga la misma división (reproducibilidad)
# stratify=y garantiza que ambos conjuntos tengan la misma proporción de riesgo (84.6% / 15.4%)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para prueba, 80% para entrenamiento
    random_state=42,    # semilla fija para reproducibilidad
    stratify=y          # mantiene la proporción de clases en ambos conjuntos
)

print("=" * 50)
print("DIVISIÓN TRAIN / TEST")
print("=" * 50)

print(f"\n📚 Entrenamiento (train):")
print(f"   Filas: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}% del total)")
print(f"   Con riesgo: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"   Sin riesgo: {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)")

print(f"\n🧪 Prueba (test):")
print(f"   Filas: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}% del total)")
print(f"   Con riesgo: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
print(f"   Sin riesgo: {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)")

print(f"\n✅ Estratificación correcta: la proporción de clases")
print(f"   se mantiene igual en train y test")

DIVISIÓN TRAIN / TEST

📚 Entrenamiento (train):
   Filas: 519 (80% del total)
   Con riesgo: 80 (15.4%)
   Sin riesgo: 439 (84.6%)

🧪 Prueba (test):
   Filas: 130 (20% del total)
   Con riesgo: 20 (15.4%)
   Sin riesgo: 110 (84.6%)

✅ Estratificación correcta: la proporción de clases
   se mantiene igual en train y test


## 6. Guardado del dataset procesado

In [10]:
# Creamos la carpeta data/processed si no existe
import os
os.makedirs("data/processed", exist_ok=True)

# Guardamos el dataset completo codificado (para el dashboard)
df_codificado.to_csv("data/processed/student_procesado.csv", index=False)

# Guardamos también los conjuntos de train y test por separado
X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

print("=" * 50)
print("ARCHIVOS GUARDADOS EN data/processed/")
print("=" * 50)
print(f"\n✅ student_procesado.csv  → dataset completo codificado")
print(f"   Filas: {df_codificado.shape[0]}, Columnas: {df_codificado.shape[1]}")
print(f"\n✅ X_train.csv → variables de entrada para entrenamiento")
print(f"   Filas: {X_train.shape[0]}, Columnas: {X_train.shape[1]}")
print(f"\n✅ X_test.csv  → variables de entrada para prueba")
print(f"   Filas: {X_test.shape[0]}, Columnas: {X_test.shape[1]}")
print(f"\n✅ y_train.csv → variable objetivo para entrenamiento")
print(f"\n✅ y_test.csv  → variable objetivo para prueba")

print(f"\n🎯 Pipeline de preparación completado exitosamente")
print(f"   Los datos están listos para el entrenamiento del modelo (Notebook 03)")

ARCHIVOS GUARDADOS EN data/processed/

✅ student_procesado.csv  → dataset completo codificado
   Filas: 649, Columnas: 32

✅ X_train.csv → variables de entrada para entrenamiento
   Filas: 519, Columnas: 31

✅ X_test.csv  → variables de entrada para prueba
   Filas: 130, Columnas: 31

✅ y_train.csv → variable objetivo para entrenamiento

✅ y_test.csv  → variable objetivo para prueba

🎯 Pipeline de preparación completado exitosamente
   Los datos están listos para el entrenamiento del modelo (Notebook 03)
